# Experiment Tracking
Compare Model A and Model B variants across hyperparameter settings.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score

sns.set_theme(style='whitegrid')
PROCESSED = '../data/processed'

## Load Processed Data

In [ ]:
def lp(f):
    with open(os.path.join(PROCESSED, f), 'rb') as fh:
        return pickle.load(fh)

X_ohe = lp('ohe_matrices.pkl')       # (X_train, X_val, X_test)
labels = lp('labels.pkl')            # (y_train, y_val, y_test, le)
X_train_ohe, X_val_ohe, X_test_ohe = X_ohe
y_train, y_val, y_test, le = labels
print('Data loaded.')

## Model A — Hyperparameter Sweep (Logistic Regression C)

In [ ]:
from sklearn.linear_model import LogisticRegression

C_values = [0.01, 0.1, 1.0, 5.0, 10.0]
results = []

for C in C_values:
    model = LogisticRegression(C=C, max_iter=500, solver='lbfgs',
                               multi_class='multinomial', n_jobs=-1)
    model.fit(X_train_ohe, y_train)
    preds = model.predict(X_val_ohe)
    acc = accuracy_score(y_val, preds)
    f1  = f1_score(y_val, preds, average='macro')
    results.append({'C': C, 'Accuracy': acc, 'Macro F1': f1})
    print(f'C={C:<6}  Acc={acc:.4f}  F1={f1:.4f}')

df_res = pd.DataFrame(results)
df_res.plot(x='C', y=['Accuracy','Macro F1'], marker='o', figsize=(8,4),
            title='LR Hyperparameter Sweep (C)')
plt.xscale('log')
plt.tight_layout()
plt.show()

## Model A — Comparison Table

In [ ]:
import joblib

MODEL_A = '../models/model_a/traditional'
model_names = ['logistic_regression', 'linear_svm', 'naive_bayes',
               'random_forest', 'stacking_ensemble']

comparison = []
for name in model_names:
    path = os.path.join(MODEL_A, f'{name}.pkl')
    if not os.path.exists(path):
        print(f'Skipping {name} (not trained yet)')
        continue
    model = joblib.load(path)
    X_v = X_val_ohe if name == 'naive_bayes' else X_val_ohe  # adjust if combined feats used
    preds = model.predict(X_v)
    comparison.append({
        'Model':    name,
        'Accuracy': round(accuracy_score(y_val, preds), 4),
        'Macro F1': round(f1_score(y_val, preds, average='macro'), 4),
    })

pd.DataFrame(comparison).sort_values('Macro F1', ascending=False)

## Clustering Evaluation (K-Means)

In [ ]:
from src.evaluate import clustering_purity, silhouette
import joblib

km_path = os.path.join(MODEL_A, 'kmeans.pkl')
if os.path.exists(km_path):
    km = joblib.load(km_path)
    cluster_labels = km.predict(X_val_ohe)
    purity = clustering_purity(y_val, cluster_labels)
    sil    = silhouette(X_val_ohe, cluster_labels)
    print(f'K-Means  Purity={purity}  Silhouette={sil}')
else:
    print('K-Means model not found. Run model_a_train.py first.')